## model training notebook 
#### this note book trains models for each train test paire and saves the result

In [ ]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import StratifiedKFold, train_test_split
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
for i in range(1, 6):
    fold_path = f"/imputed-new/Fold_{i}"
    
    # change into fold directoryf
    
    for version in range(1, 6):
        traindf = pd.read_spss(f"./Fold_{i}/Train_Imputed_m{version}.sav")
        testdf  = pd.read_spss(f"./Fold_{i}/Test_Imputed_m{version}.sav")
        traindf.columns=['Mean_Calcium', 'Mean_Phosphor', 'Mean_Magnesium', 'Age', 'BMI',
       'Apache', 'Sofa', 'Status', 'Sex', 'Smoky', 'HTN', 'DM', 'IHD',
       'CVA', 'Pulmonary_Insufficiency', 'Brain_Tumor', 'Ventilation_Status',
       'Diagnosis', 'Set_Type']
        testdf.columns=['Mean_Calcium', 'Mean_Phosphor', 'Mean_Magnesium', 'Age', 'BMI',
       'Apache', 'Sofa', 'Status', 'Sex', 'Smoky', 'HTN', 'DM', 'IHD',
       'CVA', 'Pulmonary_Insufficiency', 'Brain_Tumor', 'Ventilation_Status',
       'Diagnosis', 'Set_Type']     
        print(f"Processing Fold_{i}, m{version}")

        
        # %%
        traindf = traindf[['Mean_Calcium', 'Mean_Phosphor', 'Mean_Magnesium', 'Age', 'BMI', 'Sofa',
               'Apache', 'Status', 'Sex', 'Smoky', 'HTN', 'DM', 'IHD', 'CVA',
               'Pulmonary_Insufficiency', 'Brain_Tumor', 'Ventilation_Status',
               'Diagnosis']]
        testdf = testdf[['Mean_Calcium', 'Mean_Phosphor', 'Mean_Magnesium', 'Age', 'BMI', 'Sofa',
               'Apache', 'Status', 'Sex', 'Smoky', 'HTN', 'DM', 'IHD', 'CVA',
               'Pulmonary_Insufficiency', 'Brain_Tumor', 'Ventilation_Status',
               'Diagnosis']]
        
        # %%
        traindf.dtypes
        
        # %%
        traindf['Sex'] = traindf['Sex'].astype(float) 
        testdf['Sex'] = testdf['Sex'].astype(float) 
        
        traindf ['Status'] = traindf ['Status'].astype(float) 
        testdf ['Status'] = testdf ['Status'].astype(float) 
        
        traindf['Sex'] = traindf['Sex'] - 1
        testdf['Sex'] = testdf['Sex'] - 1 
        
        traindf['Status'] = 2 - traindf['Status'] 
        testdf['Status'] = 2 - testdf['Status'] 
        
        traindf['Sex'] = traindf['Sex'].astype('category')
        testdf['Sex'] = testdf['Sex'].astype('category')
        
        traindf['Status'] = traindf['Status'].astype('int')
        testdf['Status'] = testdf['Status'].astype('int')
        
        # %%
        traindf.dtypes
        
        # %%
        train_df_org = traindf.copy()
        test_df_org = testdf.copy() 
        
        # %%
        import numpy as np
        import pandas as pd
        from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
        from sklearn.compose import ColumnTransformer
        from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
        from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
        from sklearn.linear_model import LogisticRegression
        from sklearn.svm import SVC
        from sklearn.calibration import CalibratedClassifierCV
        from sklearn.metrics import (classification_report, roc_auc_score,
                                      average_precision_score, brier_score_loss)
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline
        import xgboost as xgb
        import warnings
        warnings.filterwarnings("ignore")
        def generate_processed_data(traindf,testdf):
        # ─────────────────────────────────────────────
        # ─────────────────────────────────────────────
        # ## preparing the data 
            X_train = traindf.drop(columns=['Status'])
            y_train = traindf['Status']
            X_test = testdf.drop(columns=['Status'])
            y_test = testdf['Status']
            # Identify columns based on dtypes in the training set
            numerical_cols = X_train.select_dtypes(include=['number']).columns
            categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns
        
            # Split the dataframes
            X_train_num = X_train[numerical_cols]
            X_train_cat = X_train[categorical_cols]
        
        
            # Initialize the preprocessors
            minmax_scaler = MinMaxScaler()
            onehot_encoder = OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False)
        
            # Fit the scaler on the numerical columns of the training set
            X_train_scaled = minmax_scaler.fit_transform(X_train_num)
        
            # Fit the encoder on the categorical columns of the training set
            X_train_encoded = onehot_encoder.fit_transform(X_train_cat)
        
            # Convert the results back to DataFrames for easy merging and inspection
            X_train_scaled = pd.DataFrame(X_train_scaled, columns=numerical_cols, index=X_train.index)
        
            # Get the new column names from the OHE
            encoded_col_names = onehot_encoder.get_feature_names_out(input_features=categorical_cols)
            X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoded_col_names, index=X_train.index)
        
            # Final combined training set
            X_train_processed = pd.concat([X_train_scaled, X_train_encoded], axis=1)
        
            print("\nProcessed Training Data Shape:", X_train_processed.shape) 
        
        
            # 1. Separate numerical and categorical columns in X_test
            X_test_num = X_test[numerical_cols]
            X_test_cat = X_test[categorical_cols]
        
            # 2. Transform numerical columns
            X_test_scaled = minmax_scaler.transform(X_test_num)
            X_test_scaled = pd.DataFrame(X_test_scaled, columns=numerical_cols, index=X_test.index)
        
            # 3. Transform categorical columns
            X_test_encoded = onehot_encoder.transform(X_test_cat)
            X_test_encoded = pd.DataFrame(X_test_encoded, columns=encoded_col_names, index=X_test.index)
        
            # 4. Combine
            X_test_processed = pd.concat([X_test_scaled, X_test_encoded], axis=1)
        
            print("Processed Test Data Shape:", X_test_processed.shape)
        
        
            y_train =y_train.astype(int) 
            y_test = y_test.astype(int) 
        
            # Imbalance ratio → scale_pos_weight for tree models, class_weight for others
            neg, pos = np.bincount(y_train)
            scale_pos_weight = neg / pos
            return X_train_processed,y_train,X_test_processed,y_test,scale_pos_weight
        
        X_train_processed,y_train,X_test_processed,y_test,scale_pos_weight = generate_processed_data(traindf=traindf,testdf=testdf)
        print(f"scale_pos_weight    →  {scale_pos_weight:.3f}\n")
        
        # %%
        
        # ─────────────────────────────────────────────
        # 0. Config
        # ─────────────────────────────────────────────
        MODEL_CHOICE = "xgboost"   # "random_forest" | "logistic_regression" | "xgboost" | "svm"
        RANDOM_STATE = 42
        N_SPLITS     = 5
        N_ITER       = 30          # RandomizedSearchCV iterations
        
        
        # ─────────────────────────────────────────────
        # 2. Model + param grid definitions
        # ─────────────────────────────────────────────
        cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        
        def get_model_and_grid(choice):
            """Return (estimator_or_pipeline, param_grid, needs_scaling, supports_early_stopping)."""
        
            if choice == "random_forest":
                model = RandomForestClassifier(
                    class_weight={0: 1, 1: scale_pos_weight},
                    random_state=RANDOM_STATE, n_jobs=-1
                )
                grid = {
                    "n_estimators":      [100, 200, 300, 500],
                    "max_depth":         [4, 6, 8, None],
                    "min_samples_split": [2, 5, 10],
                    "min_samples_leaf":  [1, 2, 4],
                    "max_features":      ["sqrt", "log2", 0.5],
                    "bootstrap":         [True, False],
                }
                return model, grid
        
            elif choice == "logistic_regression":
                model = Pipeline([
                    ("clf", LogisticRegression(
                        class_weight={0: 1, 1: scale_pos_weight},
                        max_iter=2000, random_state=RANDOM_STATE, solver="saga"
                    ))
                ])
                grid = {
                    "clf__C":       [0.001, 0.01, 0.1, 1, 10, 100],
                    "clf__penalty": ["l1", "l2", "elasticnet"],
                    "clf__l1_ratio": [0.1, 0.5, 0.9],   # only used with elasticnet
                }
                return model, grid   # scaling inside pipeline
        
            elif choice == "xgboost":
                model = xgb.XGBClassifier(
                    scale_pos_weight=scale_pos_weight,
                    eval_metric="aucpr",
                    use_label_encoder=False,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    verbosity=0,
                )
                grid = {
                    "n_estimators":     [100, 200, 300, 500],
                    "max_depth":        [3, 4, 5, 6],
                    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
                    "subsample":        [0.6, 0.8, 1.0],
                    "colsample_bytree": [0.6, 0.8, 1.0],
                    "min_child_weight": [1, 3, 5],
                    "gamma":            [0, 0.1, 0.3],
                    "reg_alpha":        [0, 0.1, 1],
                    "reg_lambda":       [1, 1.5, 2],
                }
                return model, grid
        
            elif choice == "svm":
                model = Pipeline([
                    ("clf", SVC(
                        class_weight={0: 1, 1: scale_pos_weight},
                        probability=True,          # needed for calibration
                        random_state=RANDOM_STATE
                    ))
                ])
                grid = {
                    "clf__C":      [0.01, 0.1, 1, 10, 100],
                    "clf__kernel": ["rbf", "poly", "sigmoid"],
                    "clf__gamma":  ["scale", "auto", 0.001, 0.01],
                }
                return model, grid
        
            else:
                raise ValueError(f"Unknown model: {choice}")
        
        
        
        
        
        # ─────────────────────────────────────────────
        # 6. Retrain final model with best params
        # ─────────────────────────────────────────────
        def build_final_model(choice, params, n_est_override=None):
            p = dict(params)
        
            if choice == "random_forest":
                if n_est_override:
                    p["n_estimators"] = n_est_override
                return RandomForestClassifier(
                    **p,
                    class_weight={0: 1, 1: scale_pos_weight},
                    random_state=RANDOM_STATE, n_jobs=-1
                )
        
            elif choice == "logistic_regression":
                clf_params = {k.replace("clf__", ""): v for k, v in p.items()}
                return Pipeline([
                    ("clf", LogisticRegression(
                        **clf_params,
                        class_weight={0: 1, 1: scale_pos_weight},
                        max_iter=2000, random_state=RANDOM_STATE, solver="saga"
                    ))
                ])
        
            elif choice == "xgboost":
                if n_est_override:
                    p["n_estimators"] = n_est_override
                return xgb.XGBClassifier(
                    **p,
                    scale_pos_weight=scale_pos_weight,
                    eval_metric="aucpr",
                    use_label_encoder=False,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    verbosity=0,
                )
        
            elif choice == "svm":
                clf_params = {k.replace("clf__", ""): v for k, v in p.items()}
                return Pipeline([
                    ("clf", SVC(
                        **clf_params,
                        class_weight={0: 1, 1: scale_pos_weight},
                        random_state=RANDOM_STATE
                    ))
                ])
        
        # %%
        
        # ─────────────────────────────────────────────
        # 4. Hyper-parameter search
        # ─────────────────────────────────────────────
        model, param_grid  = get_model_and_grid(MODEL_CHOICE)
        
        print(f"{'='*55}")
        print(f"  Model: {MODEL_CHOICE.upper()}")
        print(f"{'='*55}\n")
        
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,
            n_iter=N_ITER,
            scoring="roc_auc",   # good for imbalanced data
            cv=cv,
            refit=False,                   # we'll retrain manually
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=1,
        )
        search.fit(X_train_processed, y_train)
        
        xgb_best_params = search.best_params_
        print(f"\nBest hyper-parameters:\n{xgb_best_params}")
        print(f"Best CV average-precision: {search.best_score_:.4f}\n")
        final_base = build_final_model('xgboost',xgb_best_params)

        calibrated_model = CalibratedClassifierCV(
            estimator=final_base,
            method="sigmoid",   
            cv=4,
        )
        calibrated_model.fit(X_train_processed, y_train)
        print("Probability calibration done.\n")
        
        
        # ─────────────────────────────────────────────
        # 8. Evaluation on test set
        # ─────────────────────────────────────────────
        y_prob = calibrated_model.predict_proba(X_test_processed)[:, 1]
        y_prob_train = calibrated_model.predict_proba(X_train_processed)[:, 1]

        # %%
        
        results = pd.DataFrame({
            'y_true': y_test,
            'y_prob_test': y_prob,
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_xgb_test{version}.csv', index=False)
        
        results = pd.DataFrame({
            'y_true': y_train,
            'y_prob_train':y_prob_train
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_xgb_train{version}.csv', index=False)
        
        # # --- Usage ---
        # summary_df = evaluate_model(y_train, y_prob_train, y_test, y_prob, model_name="xgb")
        # print(summary_df)
        # # summary_df.to_csv("evaluation_etrics.csv", index=False)
        
        
        # %%
        # roc_auc_score(y_test,y_prob)
        
        # %%
        import numpy as np
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import brier_score_loss
        
        def calculate_calibration_metrics(y_true, y_prob):
            """
            Calculates Brier Score, Calibration Intercept, and Calibration Slope.
        
            Args:
                y_true (array-like): True binary labels (0s and 1s).
                y_prob (array-like): Predicted probabilities for the positive class.
        
            Returns:
                dict: A dictionary containing the calculated metrics.
            """
            y_true = np.asarray(y_true)
            y_prob = np.asarray(y_prob)
        
            # 1. Brier Score
            brier = brier_score_loss(y_true, y_prob)
        
            # 2. Calibration Intercept and Slope
            # Clip probabilities to avoid log(0) errors
            epsilon = 1e-15
            y_prob_clipped = np.clip(y_prob, epsilon, 1 - epsilon)
            
            # Calculate logit of probabilities
            logit_p = np.log(y_prob_clipped / (1 - y_prob_clipped)).reshape(-1, 1)
        
            # Fit a calibration model (logistic regression on logits)
            # Using a high C value to avoid regularization interfering with the measurement
            calib_model = LogisticRegression(C=1e6, solver='lbfgs', max_iter=1000)
            try:
                calib_model.fit(logit_p, y_true)
                intercept = calib_model.intercept_[0]
                slope = calib_model.coef_[0][0]
            except Exception as e:
                print(f"Could not fit calibration model: {e}")
                intercept = np.nan
                slope = np.nan
        
            return {
                "brier_score": brier,
                "calibration_intercept": intercept,
                "calibration_slope": slope
            }
        
        
        
        # Calculate the metrics
        
        # print(f"Brier Score: {metrics['brier_score']:.4f}")
        # print(f"Calibration Intercept: {metrics['calibration_intercept']:.4f} (Ideal: 0.0)")
        # print(f"Calibration Slope: {metrics['calibration_slope']:.4f} (Ideal: 1.0)")
        
        
        
        
        # %%
        MODEL_CHOICE =  "random_forest" #| "logistic_regression" | "xgboost" | "svm"
        # ─────────────────────────────────────────────
        # 4. Hyper-parameter search
        # ─────────────────────────────────────────────
        model, param_grid  = get_model_and_grid(MODEL_CHOICE)
        
        print(f"{'='*55}")
        print(f"  Model: {MODEL_CHOICE.upper()}")
        print(f"{'='*55}\n")
        
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,
            n_iter=N_ITER,
            scoring="roc_auc",   # good for imbalanced data
            cv=cv,
            refit=False,                   # we'll retrain manually
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=1,
        )
        search.fit(X_train_processed, y_train)
        
        best_params_rf = search.best_params_
        print(f"\nBest hyper-parameters:\n{best_params_rf}")
        print(f"Best CV average-precision: {search.best_score_:.4f}\n") 
        
        
        # %%
        final_base = build_final_model('random_forest',best_params_rf)
        final_base.fit(X_train_processed, y_train) 
        print("Final model trained.\n")
        

        calibrated_model = CalibratedClassifierCV(
            estimator=final_base,
            method="sigmoid",   
            cv=4,
        )
        calibrated_model.fit(X_train_processed, y_train)
        print("Probability calibration done.\n")
        

        y_prob = calibrated_model.predict_proba(X_test_processed)[:, 1]
        y_prob_train = calibrated_model.predict_proba(X_train_processed)[:, 1]
        
        results = pd.DataFrame({
            'y_true': y_test,
            'y_prob_test': y_prob,
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_rf_test{version}.csv', index=False)
        
        results = pd.DataFrame({
            'y_true': y_train,
            'y_prob_train':y_prob_train
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_rf_train{version}.csv', index=False)
        
        
        
        # %%
        roc_auc_score(y_test,y_prob) 
        
        # %%
        MODEL_CHOICE =   "logistic_regression" #| "xgboost" | "svm"
        # ─────────────────────────────────────────────
        # 4. Hyper-parameter search
        # ─────────────────────────────────────────────
        model, param_grid  = get_model_and_grid(MODEL_CHOICE)
        
        print(f"{'='*55}")
        print(f"  Model: {MODEL_CHOICE.upper()}")
        print(f"{'='*55}\n")
        
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,
            n_iter=N_ITER,
            scoring="roc_auc",   # good for imbalanced data
            cv=cv,
            refit=False,                   # we'll retrain manually
        #     n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=1,
        )
        search.fit(X_train_processed, y_train)
        
        best_params_lr = search.best_params_
        print(f"\nBest hyper-parameters:\n{best_params_lr}")
        print(f"Best CV average-precision: {search.best_score_:.4f}\n")
        
        
        
        # %%
        
        final_base = build_final_model('logistic_regression',best_params_lr )
        final_base.fit(X_train_processed, y_train)
        print("Final model trained.\n")
  
        calibrated_model = CalibratedClassifierCV(
            estimator=final_base,
            method="sigmoid",   # change to "sigmoid" if calibration set is small
            cv=4,
        )
        calibrated_model.fit(X_train_processed, y_train)
        print("Probability calibration done.\n")
        
        
        y_prob = calibrated_model.predict_proba(X_test_processed)[:, 1]
        y_prob_train = calibrated_model.predict_proba(X_train_processed)[:, 1]
        
        results = pd.DataFrame({
            'y_true': y_test,
            'y_prob_test': y_prob,
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_lr_test{version}.csv', index=False)
        
        results = pd.DataFrame({
            'y_true': y_train,
            'y_prob_train':y_prob_train
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_lr_train{version}.csv', index=False)
        # # --- Usage ---
        
        
        # %%
        
        # %%
        # Calculate the metrics
        metrics = calculate_calibration_metrics(y_test, y_prob)
        
        print(f"Brier Score: {metrics['brier_score']:.4f}")
        print(f"Calibration Intercept: {metrics['calibration_intercept']:.4f} (Ideal: 0.0)")
        print(f"Calibration Slope: {metrics['calibration_slope']:.4f} (Ideal: 1.0)")
        
        
        # %%
        MODEL_CHOICE =   "svm"
        # ─────────────────────────────────────────────
        # 4. Hyper-parameter search
        # ─────────────────────────────────────────────
        model, param_grid  = get_model_and_grid(MODEL_CHOICE)
        
        print(f"{'='*55}")
        print(f"  Model: {MODEL_CHOICE.upper()}")
        print(f"{'='*55}\n")
        
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,
            n_iter=N_ITER,
            scoring="roc_auc",   # good for imbalanced data
            cv=cv,
            refit=False,                   # we'll retrain manually
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=1,
        )
        search.fit(X_train_processed, y_train)
        
        best_params_svm = search.best_params_
        print(f"\nBest hyper-parameters:\n{best_params_svm}")
        print(f"Best CV average-precision: {search.best_score_:.4f}\n")
        
        
        
        # %%
        
        final_base = build_final_model('svm', best_params_svm)
        final_base.fit(X_train_processed, y_train)
        print("Final model trained.\n")
        
        # ─────────────────────────────────────────────
        # 7. Probability calibration
        # ─────────────────────────────────────────────
        # Isotonic regression works well when you have enough data;
        # sigmoid (Platt scaling) is safer for smaller sets.
        calibrated_model = CalibratedClassifierCV(
            estimator=final_base,
            method="sigmoid",   # change to "sigmoid" if calibration set is small
            cv=4,
        )
        calibrated_model.fit(X_train_processed, y_train)
        print("Probability calibration done.\n")
        
        
        # ─────────────────────────────────────────────
        # 8. Evaluation on test set
        # ─────────────────────────────────────────────
        y_prob = calibrated_model.predict_proba(X_test_processed)[:, 1]
        y_prob_train = calibrated_model.predict_proba(X_train_processed)[:, 1]
        
        results = pd.DataFrame({
            'y_true': y_test,
            'y_prob_test': y_prob,
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_svm_test{version}.csv', index=False)
        
        results = pd.DataFrame({
            'y_true': y_train,
            'y_prob_train':y_prob_train
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_svm_train{version}.csv', index=False)
        # # --- Usage ---
        # summary_df = evaluate_model(y_test, y_prob)
        # print(summary_df)
        
        
        # # %%
        # roc_auc_score(y_test,y_prob)
        
        # # %%
        import itertools, copy
        import numpy as np
        import torch
        from torch import nn, optim
        from sklearn.model_selection import StratifiedKFold, train_test_split
        from sklearn.metrics import f1_score, roc_auc_score
        
        
        # ── 1. Base training function (+ BatchNorm) ──────────────────────
        def train_mlp_model(X_train, y_train, X_eval, y_eval,
                            hidden_sizes, n_epoch=10000, weight_decay=0.001, lr=0.001):
            np.random.seed(42)
            torch.manual_seed(42)
        
            class MLP(nn.Module):
                def __init__(self, n_inputs, hidden_sizes, n_outputs):
                    super().__init__()
                    self.hidden_layers = nn.ModuleList()
                    self.bn_layers      = nn.ModuleList()
                    self.hidden_layers.append(nn.Linear(n_inputs, hidden_sizes[0]))
                    self.bn_layers.append(nn.BatchNorm1d(hidden_sizes[0]))
                    for i in range(1, len(hidden_sizes)):
                        self.hidden_layers.append(nn.Linear(hidden_sizes[i-1], hidden_sizes[i]))
                        self.bn_layers.append(nn.BatchNorm1d(hidden_sizes[i]))
                    self.output_layer = nn.Linear(hidden_sizes[-1], n_outputs)
        
                def forward(self, x):
                    for layer, bn in zip(self.hidden_layers, self.bn_layers):
                        x = torch.relu(bn(layer(x)))
                    return torch.sigmoid(self.output_layer(x))
        
            model     = MLP(X_train.shape[1], hidden_sizes, 1)
            criterion = nn.BCELoss(reduction='none')
            
            # Fix 2: Using AdamW's built-in weight_decay without redundant manual penalty
            optimizer = optim.AdamW([
                {'params': [p for n,p in model.named_parameters() if 'bn' not in n]},
            ], lr=lr, weight_decay=weight_decay)
        
            def to_tensors(X, y):
                X_t = torch.tensor(X, dtype=torch.float32)
                y_t = torch.tensor(y.reshape(-1, 1), dtype=torch.float32)
                # Fix 1: Corrected inverted weights. Class 1 gets the higher weight (3.0)
                w_t = torch.where(y_t == 1, torch.tensor(1.01, dtype=torch.float32), torch.tensor(1, dtype=torch.float32))
                return X_t, y_t, w_t
        
            X_tr_t, y_tr_t, w_tr = to_tensors(X_train, y_train)
            X_ev_t, y_ev_t, w_ev = to_tensors(X_eval,  y_eval)
        
            patience, min_val_loss, best_epoch, stop_training = 20, float('inf'), 0, False
            train_loss_history, eval_loss_history = [], []
            train_f1_history,   eval_f1_history   = [], []
            train_auc_history,  eval_auc_history  = [], []
        
            for epoch in range(n_epoch):
                model.train()
                optimizer.zero_grad()
                outputs  = model(X_tr_t)
                
                # Fix 2: Removed manual L2 penalty (+ alpha * ...)
                loss = (criterion(outputs, y_tr_t) * w_tr).sum() / w_tr.sum()
                loss.backward()
                optimizer.step()
        
                model.eval()
                with torch.no_grad():
                    eval_out  = model(X_ev_t)
                    # Fix 2: Removed manual L2 penalty (+ alpha * ...)
                    eval_loss = (criterion(eval_out, y_ev_t) * w_ev).sum() / w_ev.sum()
        
                    train_probs = outputs.cpu().numpy()
                    eval_probs  = eval_out.cpu().numpy()
                    train_preds = (train_probs >= 0.5).astype(int).flatten()
                    eval_preds  = (eval_probs  >= 0.5).astype(int).flatten()
        
                    train_f1  = f1_score(y_train.astype(int), train_preds, zero_division=0)
                    eval_f1   = f1_score(y_eval.astype(int),  eval_preds,  zero_division=0)
                    train_auc = roc_auc_score(y_train.astype(int), train_probs)
                    eval_auc  = roc_auc_score(y_eval.astype(int),  eval_probs)
        
                train_loss_history.append(loss.item());      eval_loss_history.append(eval_loss.item())
                train_f1_history.append(train_f1);           eval_f1_history.append(eval_f1)
                train_auc_history.append(train_auc);         eval_auc_history.append(eval_auc)
        
                if eval_loss.item() < min_val_loss:
                    min_val_loss = eval_loss.item()
                    best_epoch   = epoch
                    torch.save(model.state_dict(), 'best_model.pth')
                elif epoch - best_epoch > patience:
                    stop_training = True
        
                # if (epoch + 1) % 100 == 0:
                #     # print(f'Epoch [{epoch+1}/{n_epoch}]  Loss: {loss.item():.4f}  '
                #           # f'Eval: {eval_loss.item():.4f}  AUC: {train_auc:.4f}/{eval_auc:.4f}  '
                #           # f'F1: {train_f1:.4f}/{eval_f1:.4f}')
        
                if stop_training:
                    print(f'Early stopping at epoch {epoch+1}')
                    model.load_state_dict(torch.load('best_model.pth'))
                    break
                
            return model, train_loss_history, eval_loss_history, \
                   train_auc_history, eval_auc_history, train_f1_history, eval_f1_history
        
        # %%
        
        
        
        # ── 2. Wrapper: CV hyperparam search → retrain → y_prob ───────────────────────
        def find_best_mlp(X_train, y_train, X_test,
                          param_grid=None, n_epoch=10000):
            X_train = np.array(X_train)
            y_train = np.array(y_train)
            X_test  = np.array(X_test)
            if param_grid is None:
                param_grid = {
                    'hidden_sizes': [[32],[64],[16,32],[32,64]],
                    'lr':           [0.01, 0.001, 0.0005],
                    # Renamed 'alpha' to 'weight_decay' to reflect its true purpose
                    'weight_decay': [0.001, 0.01, 0.0001],}
        
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            best_loss, best_params = float('inf'), None
        
            for hs, lr, wd in itertools.product(
                    param_grid['hidden_sizes'], param_grid['lr'], param_grid['weight_decay']):
                fold_losses = []
                for tr_idx, val_idx in skf.split(X_train, y_train):
                    _, _, ev_loss, *_ = train_mlp_model(
                        X_train[tr_idx], y_train[tr_idx],
                        X_train[val_idx], y_train[val_idx],
                        hidden_sizes=hs, n_epoch=n_epoch, weight_decay=wd, lr=lr)
                    fold_losses.append(min(ev_loss)) # best val loss for this fold
                mean_loss = np.mean(fold_losses)
                # print(f"hs={hs} lr={lr} weight_decay={wd}  cv_loss={mean_loss:.4f}")
                if mean_loss < best_loss:
                    best_loss, best_params = mean_loss, {'hidden_sizes': hs, 'lr': lr, 'weight_decay': wd}
        
            # print(f"\nBest params: {best_params}  (cv_loss={best_loss:.4f})")
        
            # Fix 3: Stratified split for the final early-stopping holdout instead of simple slice
            X_train_fin, X_val_fin, y_train_fin, y_val_fin = train_test_split(
                X_train, y_train, test_size=0.1, stratify=y_train, random_state=42
            )
        
            final_model, *_ = train_mlp_model(
                X_train_fin, y_train_fin,
                X_val_fin, y_val_fin,
                n_epoch=n_epoch, **best_params)
        
            final_model.eval()
            with torch.no_grad():
                y_prob = final_model(
                    torch.tensor(X_test, dtype=torch.float32)
                ).cpu().numpy().flatten()
                y_prob_train = final_model(
                    torch.tensor(X_train, dtype=torch.float32)
                ).cpu().numpy().flatten()
        
            return y_prob_train, y_prob, best_params
        
        
        # %%
        y_prob_train, y_prob, best_params = find_best_mlp(X_train = X_train_processed, y_train=y_train, X_test = X_test_processed,
                          param_grid=None, n_epoch=10000) 
        
        # %%
        best_params_nn = best_params 
        
        # %%
        X_train_fin, X_val_fin, y_train_fin, y_val_fin = train_test_split(
                X_train_processed.values, y_train.values, test_size=0.1, stratify=y_train, random_state=42
            )
        
        final_model, *_ = train_mlp_model(
                X_train_fin, y_train_fin,
                X_val_fin, y_val_fin,
                n_epoch=10000, **best_params)
        
        final_model.eval()
        with torch.no_grad():
                y_prob = final_model(
                    torch.tensor(X_test_processed.values, dtype=torch.float32)
                ).cpu().numpy().flatten()
                y_prob_train = final_model(
                    torch.tensor(X_train_processed.values, dtype=torch.float32)
                ).cpu().numpy().flatten()
        
        # %%
        roc_auc_score(y_test,y_prob) 
        
        # %%
        
        results = pd.DataFrame({
            'y_true': y_test,
            'y_prob_test': y_prob,
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_nn_test{version}.csv', index=False)
        
        results = pd.DataFrame({
            'y_true': y_train,
            'y_prob_train':y_prob_train
        })
        metrics = calculate_calibration_metrics(y_test, y_prob)

        print(f"Brier Score: {metrics['brier_score']:.4f}")
        print(f"Calibration Intercept: {metrics['calibration_intercept']:.4f} (Ideal: 0.0)")
        print(f"Calibration Slope: {metrics['calibration_slope']:.4f} (Ideal: 1.0)")
        
        
        results.to_csv(f'./working/Fold_{i}_predictions_nn_train{version}.csv', index=False)
        
        # %%
        def prepare_data_BBN(df):
            
            df["Mean_Magnesium_CAT"] = pd.cut(
            df["Mean_Magnesium"],
                bins=[float('-inf'),1.8,2.4,float('inf')],
                labels=[1,2,3],
                right=True
            )
            
            df["Mean_Phosphor_CAT"] = pd.cut(
                df["Mean_Phosphor"],
                bins=[float('-inf'),2.5,4.5,float('inf')],
                labels=[1,2,3],
                right=True
            )
        
            df["Mean_Calcium_CAT"] = pd.cut(
                df["Mean_Calcium"],
                bins=[float('-inf'),8.2,9.3,float('inf')],
                labels=[1,2,3],
                right=True
            )
        
            df["Apache_CAT"] = pd.cut(
                df["Apache"],
                bins=[float('-inf'),15,24,float('inf')],
                labels=[1,2,3],
                right=True
            )
        
            df["BMI_CAT"] = pd.cut(
                df["BMI"],
                bins=[float('-inf'),18.5,24.9,29.9,float('inf')],
                labels=[1,2,3,4],
                right=True
            )
        
            df["Sofa_CAT"] = pd.cut(
                df["Sofa"],
                bins=[float('-inf'),4,7,10,float('inf')],
                labels=[1,2,3,4],
                right=True
            )
        
            df["Age_CAT"] = pd.cut(
                df["Age"],
                bins=[float('-inf'),44,64,79,float('inf')],
                labels=[1,2,3,4],
                right=True
            )
        
            df = df.drop(columns=['Mean_Calcium','Mean_Phosphor','Mean_Magnesium','BMI','Apache','Sofa','Age'])
            return df.astype('object')
        
        # %%
        train = prepare_data_BBN(train_df_org.copy())
        test =  prepare_data_BBN(test_df_org.copy())
        
        # %%
        import numpy as np
        import pandas as pd
        from pgmpy.estimators import HillClimbSearch, BDeu, BayesianEstimator, ExpertKnowledge
        from pgmpy.models import DiscreteBayesianNetwork
        train = prepare_data_BBN(train_df_org.copy())
        test =  prepare_data_BBN(test_df_org.copy())
        # Variables setup
        CLASS_NODE = 'Status'
        FIXED = ['Sex', 'Age_CAT', 'Smoky']
        CHRONIC = ['HTN', 'DM', 'IHD', 'CVA', 'Brain_Tumor', 'BMI_CAT']
        ACUTE = ['Sofa_CAT', 'Apache_CAT', 'Mean_Calcium_CAT', 'Mean_Phosphor_CAT', 'Mean_Magnesium_CAT',
                 'Ventilation_Status', 'Pulmonary_Insufficiency', 'Diagnosis']
        ALL_FEATURES = FIXED + CHRONIC + ACUTE
        ALL_NODES = ALL_FEATURES + [CLASS_NODE]
        
        # Forbidden Edges
        forbidden_edges = []
        for src in ALL_NODES:
            for trg in FIXED: 
                if src != trg: forbidden_edges.append((src, trg))
            for trg in CHRONIC:
                if src in ACUTE: forbidden_edges.append((src, trg))
            forbidden_edges.append((CLASS_NODE, src))
        
        expert_knowledge = ExpertKnowledge(nodes=ALL_NODES, forbidden_edges=forbidden_edges)
        
        print("--- Training Model on Entire Train Set ---")
        
        # 1. Structure Learning
        hc = HillClimbSearch(data=train)
        dag = hc.estimate(
            scoring_method=BDeu(data=train, equivalent_sample_size=10), 
            expert_knowledge=expert_knowledge
        )
        
        # 2. State Names & Non-Uniform Prior Setup
        state_names = {node: sorted(train[node].dropna().unique().tolist()) for node in ALL_NODES}
        status_prior = [7, 3] if state_names[CLASS_NODE][0] == 0 else [3, 7] 
        
        pseudo_counts = {}
        for node in ALL_NODES:
            n_states = len(state_names[node])
            parents = list(dag.get_parents(node))
            n_comb = np.prod([len(state_names[p]) for p in parents]) if parents else 1
            
            if node == CLASS_NODE:
                counts = np.tile(np.array(status_prior).reshape(-1, 1), n_comb)
            else:
                counts = np.full((n_states, int(n_comb)), 10 / n_states)
            pseudo_counts[node] = counts
        
        # 3. Parameter Learning
        model = DiscreteBayesianNetwork(dag.edges())
        model.add_nodes_from(ALL_NODES) 
        
        model.fit(
            data=train, 
            estimator=BayesianEstimator, 
            prior_type='dirichlet', 
            pseudo_counts=pseudo_counts, 
            state_names=state_names 
        )
        
        print("--- Predicting Probabilities ---")
        
        # 4. Predict on Train and Test
        train_probs_df = model.predict_probability(train[ALL_FEATURES])
        test_probs_df = model.predict_probability(test[ALL_FEATURES])
        
        # Extract probabilities for class 1
        train_y_probs = train_probs_df[f'{CLASS_NODE}_1'].values
        test_y_probs = test_probs_df[f'{CLASS_NODE}_1'].values
        
        print("Complete. 'train_y_probs' and 'test_y_probs' are ready.")
        


        train_y_probs = train_probs_df[f'{CLASS_NODE}_1'].values

        test_y_probs = test_probs_df[f'{CLASS_NODE}_1'].values

        
        # ── 5. Platt Scaling via CalibratedClassifierCV pattern ─────────
        from sklearn.calibration import calibration_curve
        from sklearn.isotonic import IsotonicRegression

        y_train_bbn = train[CLASS_NODE].astype(int).values

        # Stratified split to avoid calibrating on training data
        idx_cal_train, idx_cal_val = train_test_split(
            np.arange(len(y_train_bbn)),
            test_size=0.2,
            stratify=y_train_bbn,
            random_state=42
        )

        # Fit isotonic calibrator (more flexible than Platt for BBN output)
        calibrator = IsotonicRegression(out_of_bounds='clip')
        calibrator.fit(
            train_y_probs[idx_cal_train],
            y_train_bbn[idx_cal_train]
        )

        # Apply calibration
        train_y_probs_cal = calibrator.predict(train_y_probs)
        test_y_probs_cal  = calibrator.predict(test_y_probs)

        
        # %%
        metrics = calculate_calibration_metrics(y_test, y_prob)
        
        print(f"Brier Score: {metrics['brier_score']:.4f}")
        print(f"Calibration Intercept: {metrics['calibration_intercept']:.4f} (Ideal: 0.0)")
        print(f"Calibration Slope: {metrics['calibration_slope']:.4f} (Ideal: 1.0)")
        
        
        
        results = pd.DataFrame({
            'y_true': y_test,
            'test_y_probs': test_y_probs_cal,
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_bbn_test{version}.csv', index=False)
        
        results = pd.DataFrame({
            'y_true': y_train,
            'train_y_probs':train_y_probs_cal
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_bbn_train{version}.csv', index=False)
        
        # %%
        import numpy as np
        import torch
        from sklearn.model_selection import StratifiedKFold
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import Pipeline
        from sklearn.preprocessing import StandardScaler
        from pgmpy.models import DiscreteBayesianNetwork
        from pgmpy.estimators import HillClimbSearch, BDeu, BayesianEstimator, ExpertKnowledge
        
        CLASS_NODE = 'Status'
        FIXED = ['Sex', 'Age_CAT', 'Smoky']
        CHRONIC = ['HTN', 'DM', 'IHD', 'CVA', 'Brain_Tumor', 'BMI_CAT']
        ACUTE = ['Sofa_CAT', 'Apache_CAT', 'Mean_Calcium_CAT', 'Mean_Phosphor_CAT',
                 'Mean_Magnesium_CAT', 'Ventilation_Status', 'Pulmonary_Insufficiency', 'Diagnosis']
        ALL_FEATURES = FIXED + CHRONIC + ACUTE
        
        
        def _build_forbidden_edges():
            forbidden = []
            for src in CHRONIC + ACUTE + [CLASS_NODE]:
                for tgt in FIXED:
                    forbidden.append((src, tgt))
            for src in ACUTE:
                for tgt in CHRONIC:
                    forbidden.append((src, tgt))
            for tgt in ALL_FEATURES:
                forbidden.append((CLASS_NODE, tgt))
            return forbidden
        
        
        def _fit_bbn(train_df):
            expert_knowledge = ExpertKnowledge(nodes=ALL_NODES, forbidden_edges=forbidden_edges)
        
            # 1. Structure Learning
            hc = HillClimbSearch(data=train_df)
            dag = hc.estimate(
                scoring_method=BDeu(data=train_df, equivalent_sample_size=10), 
                expert_knowledge=expert_knowledge
            )
        
            # 2. State Names & Non-Uniform Prior Setup
            state_names = {node: sorted(train_df[node].dropna().unique().tolist()) for node in ALL_NODES}
            status_prior = [7, 3] if state_names[CLASS_NODE][0] == 0 else [3, 7] 
        
            pseudo_counts = {}
            for node in ALL_NODES:
                n_states = len(state_names[node])
                parents = list(dag.get_parents(node))
                n_comb = np.prod([len(state_names[p]) for p in parents]) if parents else 1
        
                if node == CLASS_NODE:
                    counts = np.tile(np.array(status_prior).reshape(-1, 1), n_comb)
                else:
                    counts = np.full((n_states, int(n_comb)), 10 / n_states)
                pseudo_counts[node] = counts
        
            # 3. Parameter Learning
            model = DiscreteBayesianNetwork(dag.edges())
            model.add_nodes_from(ALL_NODES) 
        
            model.fit(
                data=train_df, 
                estimator=BayesianEstimator, 
                prior_type='dirichlet', 
                pseudo_counts=pseudo_counts, 
                state_names=state_names 
            )
        
            return model
        
        
        def _bbn_predict(model, df):
            probs = model.predict_probability(df[ALL_FEATURES])
            return probs[f'{CLASS_NODE}_1'].values
        
        
        def _mlp_predict(mlp_model, X):
            mlp_model.eval()
            with torch.no_grad():
                return mlp_model(torch.tensor(X, dtype=torch.float32)).numpy().flatten()
        
        
        def build_stacking_ensemble(X_train, y_train, X_test, y_test,train_df_bbn, test_df_bbn,
                                     sklearn_models, best_mlp_params,
                                     n_splits=5, random_state=42):
            skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
            sk_names = sorted(sklearn_models.keys())
        
            bbn_oof = np.zeros(len(y_train))
            mlp_oof = np.zeros(len(y_train))
            sk_oof  = {name: np.zeros(len(y_train)) for name in sk_names}
        
            for tr_idx, val_idx in skf.split(X_train, y_train):
                X_tr, X_val = X_train[tr_idx], X_train[val_idx]
                y_tr, y_val = y_train[tr_idx], y_train[val_idx]
        
                # BBN
                bbn_oof[val_idx] = _bbn_predict(
                    _fit_bbn(train_df_bbn.iloc[tr_idx].reset_index(drop=True)),
                    train_df_bbn.iloc[val_idx].reset_index(drop=True)
                )
        
        
                mlp_model, *_ = train_mlp_model(
                    X_tr, y_tr, X_val, y_val,
                    hidden_sizes=best_mlp_params['hidden_sizes'],
                    lr=best_mlp_params['lr'],
                    weight_decay=best_mlp_params['weight_decay']
                )
                mlp_oof[val_idx] = _mlp_predict(mlp_model, X_val)
        
                # sklearn
                for name, clf in sklearn_models.items():
                    clf.fit(X_tr, y_tr)
                    sk_oof[name][val_idx] = clf.predict_proba(X_val)[:, 1]
        
            # ── refit all base models on full training data ──────────────────────────
            fitted_bbn = _fit_bbn(train_df_bbn)
        
            split = int(0.9 * len(X_train))
            fitted_mlp, *_ = train_mlp_model(
                X_train[:split], y_train[:split], X_train[split:], y_train[split:],
                hidden_sizes=best_mlp_params['hidden_sizes'],
                lr=best_mlp_params['lr'],
                weight_decay=best_mlp_params['weight_decay']
            )
        
            fitted_sklearn = {}
            for name, clf in sklearn_models.items():
                clf.fit(X_train, y_train)
                fitted_sklearn[name] = clf
        
            # ── meta-features ────────────────────────────────────────────────────────
            meta_train = np.column_stack(
                [bbn_oof, mlp_oof] + [sk_oof[n] for n in sk_names]
            )
        
            # ── meta-learner ─────────────────────────────────────────────────────────
            meta_learner = Pipeline([
                ('scaler', StandardScaler()),
                ('lr', LogisticRegression( random_state=random_state,
                        max_iter=2000))
            ])
            meta_learner.fit(meta_train, y_train)
        
            # ── test predictions (no retraining) ────────────────────────────────────
            bbn_test  = _bbn_predict(fitted_bbn, test_df_bbn)
            mlp_test  = _mlp_predict(fitted_mlp, X_test)
            sk_test   = [fitted_sklearn[n].predict_proba(X_test)[:, 1] for n in sk_names]
        
            meta_test   = np.column_stack([bbn_test, mlp_test] + sk_test)
            y_test_prob = meta_learner.predict_proba(meta_test)[:, 1]
        
            # ── train predictions (no retraining) ────────────────────────────────────
            bbn_train  = _bbn_predict(fitted_bbn, train_df_bbn)
            mlp_train  = _mlp_predict(fitted_mlp, X_train)
            sk_train   = [fitted_sklearn[n].predict_proba(X_train)[:, 1] for n in sk_names]
        
            meta_train   = np.column_stack([bbn_train, mlp_train] + sk_train)
            y_train_prob = meta_learner.predict_proba(meta_train)[:, 1]
        
            return y_train_prob, y_test_prob, meta_learner, {
                'fitted_bbn': fitted_bbn,
                'fitted_mlp': fitted_mlp,
                'fitted_sklearn': fitted_sklearn,}
        
        
        
        # %%
        best_params_lr = {k.replace("clf__", ""): v for k, v in best_params_lr.items()}
        best_params_lr
        
        # %%
        best_params_svm = {k.replace("clf__", ""): v for k, v in best_params_svm.items()}
        best_params_svm
        
        # %%
        #bbn,nn,svm,rf,xgb,lr
        y_train_prob, y_test_prob, meta_learner, fitted_models = build_stacking_ensemble(
            X_train_processed.values, y_train.values,
            X_test_processed.values, y_test.values,
            train, test,
            sklearn_models={'svm':SVC(
                        **best_params_svm,
                        class_weight={0: 1, 1: scale_pos_weight},
                        random_state=RANDOM_STATE,probability=True),
                            
                            'rf': RandomForestClassifier(**best_params_rf,
                    class_weight={0: 1, 1: scale_pos_weight},
                    random_state=RANDOM_STATE, n_jobs=-1
                ),'xgb':xgb.XGBClassifier(
                    **xgb_best_params,
                    eval_metric="aucpr",
                    use_label_encoder=False,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    verbosity=0,
                ),  'lr':LogisticRegression(
                        **best_params_lr,
                        class_weight={0: 1, 1: scale_pos_weight},
                        max_iter=2000, random_state=RANDOM_STATE, solver="saga"
                    )},
            best_mlp_params=best_params_nn
        )
        
        # %%
        roc_auc_score(y_test, y_test_prob)
        
        # %%
        import numpy as np
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import brier_score_loss
        
        def calculate_calibration_metrics(y_true, y_prob):
            """
            Calculates Brier Score, Calibration Intercept, and Calibration Slope.
        
            Args:
                y_true (array-like): True binary labels (0s and 1s).
                y_prob (array-like): Predicted probabilities for the positive class.
        
            Returns:
                dict: A dictionary containing the calculated metrics.
            """
            y_true = np.asarray(y_true)
            y_prob = np.asarray(y_prob)
        
            # 1. Brier Score
            brier = brier_score_loss(y_true, y_prob)
        
            # 2. Calibration Intercept and Slope
            # Clip probabilities to avoid log(0) errors
            epsilon = 1e-15
            y_prob_clipped = np.clip(y_prob, epsilon, 1 - epsilon)
            
            # Calculate logit of probabilities
            logit_p = np.log(y_prob_clipped / (1 - y_prob_clipped)).reshape(-1, 1)
        
            # Fit a calibration model (logistic regression on logits)
            # Using a high C value to avoid regularization interfering with the measurement
            calib_model = LogisticRegression(C=1e6, solver='lbfgs', max_iter=1000)
            try:
                calib_model.fit(logit_p, y_true)
                intercept = calib_model.intercept_[0]
                slope = calib_model.coef_[0][0]
            except Exception as e:
                print(f"Could not fit calibration model: {e}")
                intercept = np.nan
                slope = np.nan
        
            return {
                "brier_score": brier,
                "calibration_intercept": intercept,
                "calibration_slope": slope
            }
        
        
        
        # Calculate the metrics
        metrics = calculate_calibration_metrics(y_test, y_test_prob)
        
        print(f"Brier Score: {metrics['brier_score']:.4f}")
        print(f"Calibration Intercept: {metrics['calibration_intercept']:.4f} (Ideal: 0.0)")
        print(f"Calibration Slope: {metrics['calibration_slope']:.4f} (Ideal: 1.0)")
        
        
        
        
        # %%
        results = pd.DataFrame({
            'y_true': y_test,
            'test_y_probs': y_test_prob,
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_stack_test{version}.csv', index=False)
        
        results = pd.DataFrame({
            'y_true': y_train,
            'train_y_probs':y_train_prob
        })
        
        results.to_csv(f'./working/Fold_{i}_predictions_stack_train{version}.csv', index=False)
        
        # %%
        meta_learner.named_steps['lr'].coef_
        #bbn,nn,svm,rf,xgb,lr
        
        # %%
        
        
        
        


In [ ]:
!zip -r all_files_new.zip ./working

In [ ]:
import numpy as np 
import pandas as pd 
from sklearn.model_selection import StratifiedKFold, train_test_split
import os
import warnings
import shap

all_shap = []
all_X = []
feature_names = None
warnings.filterwarnings("ignore", category=UserWarning)
for i in range(1, 6):
    fold_shap_list = []
    fold_X_list = []
    fold_path = f"./Fold_{i}"
    
    # change into fold directoryf
    
    for version in range(1, 6):
        traindf = pd.read_spss(f"./Fold_{i}/Train_Imputed_m{version}.sav")
        testdf  = pd.read_spss(f"./Fold_{i}/Test_Imputed_m{version}.sav")
        traindf.columns=corrected_names = [
    'Mean Serum Calcium', 
    'Mean Serum Phosphorus', 
    'Mean Serum Magnesium', 
    'Age (years)', 
    'BMI',
    'APACHE II Score', 
    'Mean SOFA Score', 
    'ICU Mortality', 
    'Gender', 
    'Smoking Status', 
    'Hypertension', 
    'Diabetes Mellitus', 
    'Ischemic Heart Disease',
    'Cerebrovascular Accident', 
    'Pulmonary Insufficiency', 
    'Brain Tumor', 
    'Mechanical Ventilation',
    'Admission Diagnosis', 
    'Dataset Split'
]
        testdf.columns=corrected_names = [
    'Mean Serum Calcium', 
    'Mean Serum Phosphorus', 
    'Mean Serum Magnesium', 
    'Age (years)', 
    'BMI',
    'APACHE II Score', 
    'Mean SOFA Score', 
    'ICU Mortality', 
    'Gender', 
    'Smoking Status', 
    'Hypertension', 
    'Diabetes Mellitus', 
    'Ischemic Heart Disease',
    'Cerebrovascular Accident', 
    'Pulmonary Insufficiency', 
    'Brain Tumor', 
    'Mechanical Ventilation',
    'Admission Diagnosis', 
    'Dataset Split'
]      
        print(f"Processing Fold_{i}, m{version}")

        
        # %%
# 3. Define the corrected list matching your exact original column order
        corrected_cols = [
    'Mean Serum Calcium', 'Mean Serum Phosphorus', 'Mean Serum Magnesium', 
    'Age (years)', 'BMI', 'SOFA Score', 'APACHE II Score', 
    'ICU Mortality', 'Gender', 'Smoking Status', 'Hypertension', 
    'Diabetes Mellitus', 'Ischemic Heart Disease', 'Cerebrovascular Accident', 
    'Pulmonary Insufficiency', 'Brain Tumor', 'Mechanical Ventilation', 
    'Admission Diagnosis'
]

# 4. Filter dataframes by the updated names
        traindf = traindf[corrected_cols]
        testdf = testdf[corrected_cols]
        
        # %%
        traindf.dtypes
        
        # %%
        traindf['Gender'] = traindf['Gender'].astype(float) 
        testdf['Gender'] = testdf['Gender'].astype(float) 
        
        traindf ['ICU Mortality'] = traindf ['ICU Mortality'].astype(float) 
        testdf ['ICU Mortality'] = testdf ['ICU Mortality'].astype(float) 
        
        traindf['Gender'] = traindf['Gender'] - 1
        testdf['Gender'] = testdf['Gender'] - 1 
        
        traindf['ICU Mortality'] = 2 - traindf['ICU Mortality'] 
        testdf['ICU Mortality'] = 2 - testdf['ICU Mortality'] 
        
        traindf['Gender'] = traindf['Gender'].astype('category')
        testdf['Gender'] = testdf['Gender'].astype('category')
        
        traindf['ICU Mortality'] = traindf['ICU Mortality'].astype('int')
        testdf['ICU Mortality'] = testdf['ICU Mortality'].astype('int')
        
        # %%
        traindf.dtypes
        
        # %%
        train_df_org = traindf.copy()
        test_df_org = testdf.copy() 
        
        # %%
        import numpy as np
        import pandas as pd
        from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
        from sklearn.compose import ColumnTransformer
        from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
        from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
        from sklearn.linear_model import LogisticRegression
        from sklearn.svm import SVC
        from sklearn.calibration import CalibratedClassifierCV
        from sklearn.metrics import (classification_report, roc_auc_score,
                                      average_precision_score, brier_score_loss)
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline
        import xgboost as xgb
        import warnings
        warnings.filterwarnings("ignore")
        def generate_processed_data(traindf,testdf):
        # ─────────────────────────────────────────────
        # ─────────────────────────────────────────────
        # ## preparing the data 
            X_train = traindf.drop(columns=['ICU Mortality'])
            y_train = traindf['ICU Mortality']
            X_test = testdf.drop(columns=['ICU Mortality'])
            y_test = testdf['ICU Mortality']
            # Identify columns based on dtypes in the training set
            numerical_cols = X_train.select_dtypes(include=['number']).columns
            categorical_cols = X_train.select_dtypes(include=['object', 'category']).columns
        
            # Split the dataframes
            X_train_num = X_train[numerical_cols]
            X_train_cat = X_train[categorical_cols]
        
        
            # Initialize the preprocessors
            minmax_scaler = MinMaxScaler()
            onehot_encoder = OneHotEncoder(handle_unknown='ignore', drop='if_binary', sparse_output=False)
        
            # Fit the scaler on the numerical columns of the training set
            X_train_scaled = minmax_scaler.fit_transform(X_train_num)
        
            # Fit the encoder on the categorical columns of the training set
            X_train_encoded = onehot_encoder.fit_transform(X_train_cat)
        
            # Convert the results back to DataFrames for easy merging and inspection
            X_train_scaled = pd.DataFrame(X_train_scaled, columns=numerical_cols, index=X_train.index)
        
            # Get the new column names from the OHE
            encoded_col_names = onehot_encoder.get_feature_names_out(input_features=categorical_cols)
            X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoded_col_names, index=X_train.index)
        
            # Final combined training set
            X_train_processed = pd.concat([X_train_scaled, X_train_encoded], axis=1)
        
            print("\nProcessed Training Data Shape:", X_train_processed.shape) 
        
        
            # 1. Separate numerical and categorical columns in X_test
            X_test_num = X_test[numerical_cols]
            X_test_cat = X_test[categorical_cols]
        
            # 2. Transform numerical columns
            X_test_scaled = minmax_scaler.transform(X_test_num)
            X_test_scaled = pd.DataFrame(X_test_scaled, columns=numerical_cols, index=X_test.index)
        
            # 3. Transform categorical columns
            X_test_encoded = onehot_encoder.transform(X_test_cat)
            X_test_encoded = pd.DataFrame(X_test_encoded, columns=encoded_col_names, index=X_test.index)
        
            # 4. Combine
            X_test_processed = pd.concat([X_test_scaled, X_test_encoded], axis=1)
        
            print("Processed Test Data Shape:", X_test_processed.shape)
        
        
            y_train =y_train.astype(int) 
            y_test = y_test.astype(int) 
        
            # Imbalance ratio → scale_pos_weight for tree models, class_weight for others
            neg, pos = np.bincount(y_train)
            scale_pos_weight = neg / pos
            return X_train_processed,y_train,X_test_processed,y_test,scale_pos_weight
        
        X_train_processed,y_train,X_test_processed,y_test,scale_pos_weight = generate_processed_data(traindf=traindf,testdf=testdf)
        print(f"scale_pos_weight    →  {scale_pos_weight:.3f}\n")
        
        # %%
        
        # ─────────────────────────────────────────────
        # 0. Config
        # ─────────────────────────────────────────────
        MODEL_CHOICE = "xgboost"   # "random_forest" | "logistic_regression" | "xgboost" | "svm"
        RANDOM_STATE = 42
        N_SPLITS     = 5
        N_ITER       = 30          # RandomizedSearchCV iterations
        
        
        # ─────────────────────────────────────────────
        # 2. Model + param grid definitions
        # ─────────────────────────────────────────────
        cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
        
        def get_model_and_grid(choice):
            """Return (estimator_or_pipeline, param_grid, needs_scaling, supports_early_stopping)."""
        
            if choice == "random_forest":
                model = RandomForestClassifier(
                    class_weight={0: 1, 1: scale_pos_weight},
                    random_state=RANDOM_STATE, n_jobs=-1
                )
                grid = {
                    "n_estimators":      [100, 200, 300, 500],
                    "max_depth":         [4, 6, 8, None],
                    "min_samples_split": [2, 5, 10],
                    "min_samples_leaf":  [1, 2, 4],
                    "max_features":      ["sqrt", "log2", 0.5],
                    "bootstrap":         [True, False],
                }
                return model, grid
        
            elif choice == "logistic_regression":
                model = Pipeline([
                    ("clf", LogisticRegression(
                        class_weight={0: 1, 1: scale_pos_weight},
                        max_iter=2000, random_state=RANDOM_STATE, solver="saga"
                    ))
                ])
                grid = {
                    "clf__C":       [0.001, 0.01, 0.1, 1, 10, 100],
                    "clf__penalty": ["l1", "l2", "elasticnet"],
                    "clf__l1_ratio": [0.1, 0.5, 0.9],   # only used with elasticnet
                }
                return model, grid   # scaling inside pipeline
        
            elif choice == "xgboost":
                model = xgb.XGBClassifier(
                    scale_pos_weight=scale_pos_weight,
                    eval_metric="aucpr",
                    use_label_encoder=False,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    verbosity=0,
                )
                grid = {
                    "n_estimators":     [100, 200, 300, 500],
                    "max_depth":        [3, 4, 5, 6],
                    "learning_rate":    [0.01, 0.05, 0.1, 0.2],
                    "subsample":        [0.6, 0.8, 1.0],
                    "colsample_bytree": [0.6, 0.8, 1.0],
                    "min_child_weight": [1, 3, 5],
                    "gamma":            [0, 0.1, 0.3],
                    "reg_alpha":        [0, 0.1, 1],
                    "reg_lambda":       [1, 1.5, 2],
                }
                return model, grid
        
            elif choice == "svm":
                model = Pipeline([
                    ("clf", SVC(
                        class_weight={0: 1, 1: scale_pos_weight},
                        probability=True,          # needed for calibration
                        random_state=RANDOM_STATE
                    ))
                ])
                grid = {
                    "clf__C":      [0.01, 0.1, 1, 10, 100],
                    "clf__kernel": ["rbf", "poly", "sigmoid"],
                    "clf__gamma":  ["scale", "auto", 0.001, 0.01],
                }
                return model, grid
        
            else:
                raise ValueError(f"Unknown model: {choice}")
        
        
        
        
        
        # ─────────────────────────────────────────────
        # 6. Retrain final model with best params
        # ─────────────────────────────────────────────
        def build_final_model(choice, params, n_est_override=None):
            p = dict(params)
        
            if choice == "random_forest":
                if n_est_override:
                    p["n_estimators"] = n_est_override
                return RandomForestClassifier(
                    **p,
                    class_weight={0: 1, 1: scale_pos_weight},
                    random_state=RANDOM_STATE, n_jobs=-1
                )
        
            elif choice == "logistic_regression":
                clf_params = {k.replace("clf__", ""): v for k, v in p.items()}
                return Pipeline([
                    ("clf", LogisticRegression(
                        **clf_params,
                        class_weight={0: 1, 1: scale_pos_weight},
                        max_iter=2000, random_state=RANDOM_STATE, solver="saga"
                    ))
                ])
        
            elif choice == "xgboost":
                if n_est_override:
                    p["n_estimators"] = n_est_override
                return xgb.XGBClassifier(
                    **p,
                    scale_pos_weight=scale_pos_weight,
                    eval_metric="aucpr",
                    use_label_encoder=False,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    verbosity=0,
                )
        
            elif choice == "svm":
                clf_params = {k.replace("clf__", ""): v for k, v in p.items()}
                return Pipeline([
                    ("clf", SVC(
                        **clf_params,
                        class_weight={0: 1, 1: scale_pos_weight},
                        random_state=RANDOM_STATE
                    ))
                ])
        
        # %%
        
        # ─────────────────────────────────────────────
        # 4. Hyper-parameter search
        # ─────────────────────────────────────────────
        model, param_grid  = get_model_and_grid(MODEL_CHOICE)
        
        print(f"{'='*55}")
        print(f"  Model: {MODEL_CHOICE.upper()}")
        print(f"{'='*55}\n")
        
        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grid,
            n_iter=N_ITER,
            scoring="roc_auc",   # good for imbalanced data
            cv=cv,
            refit=False,                   # we'll retrain manually
            n_jobs=-1,
            random_state=RANDOM_STATE,
            verbose=1,
        )
        search.fit(X_train_processed, y_train)
        
        xgb_best_params = search.best_params_
        
        print(f"\nBest hyper-parameters:\n{xgb_best_params}")
        print(f"Best ROC-AUC: {search.best_score_:.4f}\n")

        final_model = build_final_model("xgboost", xgb_best_params)

        final_model.fit(X_train_processed, y_train)

        explainer = shap.TreeExplainer(final_model)

        shap_values = explainer.shap_values(X_test_processed)

        fold_shap_list.append(shap_values)
        fold_X_list.append(X_test_processed.values)

        feature_names = X_test_processed.columns.tolist()
    print(f"\nAveraging imputations for Fold {i}")

    mean_shap_fold = np.mean(fold_shap_list, axis=0)

    mean_X_fold = np.mean(fold_X_list, axis=0)

    all_shap.append(mean_shap_fold)

    all_X.append(
        pd.DataFrame(
            mean_X_fold,
            columns=feature_names
            )
                )
final_shap = np.vstack(all_shap)

final_X = pd.concat(
    all_X,
    axis=0,
    ignore_index=True
)

shap_df = pd.DataFrame(
    final_shap,
    columns=feature_names
)

shap_df.to_csv(
    "./working/pooled_shap_values.csv",
    index=False
)

final_X.to_csv(
    "./pooled_feature_values.csv",
    index=False
)

print("Saved:")
print("./working/pooled_shap_values.csv")
print("./working/pooled_feature_values.csv")
print("Final shape:", shap_df.shape)

In [ ]:
import pandas as pd
import shap

X = final_X

shap_values = shap_df.values

import pandas as pd
import shap
import matplotlib.pyplot as plt



# Publication settings
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "font.family": "sans-serif"
})

# Create figure
plt.figure(figsize=(8, 6))

shap.summary_plot(
    shap_values,
    X,
    max_display=20,
    cmap="viridis",      # cleaner than default red-blue
    show=False
)

plt.tight_layout()

plt.savefig(
    "SHAP_Beeswarm_XGB.tiff",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "SHAP_Beeswarm_XGB.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score, brier_score_loss
from sklearn.calibration import calibration_curve

# ==========================================
# 1. JOURNAL-STYLE TYPOGRAPHY & SIZE CONFIG
# ==========================================
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 13,                # Baselines
    'axes.labelsize': 15,           # Feature axes labels
    'axes.titlesize': 16,           # Standalone titles
    'xtick.labelsize': 12,          # Ticks
    'ytick.labelsize': 12,          # Ticks
    'legend.fontsize': 11,          # Model legends
    'legend.frameon': False,        # Clean minimalist layout
    'savefig.dpi': 600,             # Print-ready resolution
    'savefig.bbox': 'tight'
})

# Explicit professional palette for 7 models
MODEL_COLORS = {
    'stacking': '#d62728',  # Bold Crimson (The primary focus)
    'xgb': '#ff7f0e',       # Safety Orange
    'rf': '#2ca02c',        # Forest Green
    'svm': '#1f77b4',       # Soft Blue
    'lr': '#9467bd',        # Royal Purple
    'nn': '#17becf',        # Deep Teal
    'bbn': '#8c564b'        # Muted Brown
}

# ==========================================
# 2. SEPARATE PLOTTING FUNCTIONS
# ==========================================
def generate_individual_plots(csv_path):
    print(f"Reading dataset from: {csv_path}")
    df = pd.read_csv(csv_path)
    
    # Normalize model names to lowercase to map accurately to colors
    df['Model'] = df['Model'].astype(str).str.lower().str.strip()
    unique_models = df['Model'].unique()
    print(f"Detected models in CSV: {list(unique_models)}")

    fig_size = (7.5, 6.5)

    # -------------------------------------------------------------------------
    # FIGURE 1: ROC Curve
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=fig_size)
    ax.plot([0, 1], [0, 1], linestyle='--', color='#7f7f7f', alpha=0.6, linewidth=1.5, label='Chance')
    
    for model_name in unique_models:
        m_df = df[df['Model'] == model_name]
        fpr, tpr, _ = roc_curve(m_df['y_true'], m_df['y_pred'])
        roc_auc = auc(fpr, tpr)
        
        color = MODEL_COLORS.get(model_name, '#000000')
        linewidth = 2.5 if model_name == 'stacking' else 1.8
        
        ax.plot(fpr, tpr, color=color, lw=linewidth,
                label=f'{model_name.upper()} (AUC = {roc_auc:.3f})')
        
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel('False Positive Rate (1 - Specificity)', labelpad=10)
    ax.set_ylabel('True Positive Rate (Sensitivity)', labelpad=10)
    ax.set_title('Receiver Operating Characteristic (ROC)', weight='bold', pad=15)
    
    # Shifted slightly away from the right edge and up from the bottom line
    ax.legend(loc='lower right', bbox_to_anchor=(0.98, 0.08))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)
    
    plt.savefig('fig_roc_curve.png')
    plt.savefig('fig_roc_curve.pdf', format='pdf')
    plt.close(fig)
    print("Saved: fig_roc_curve.png/.pdf")

    # -------------------------------------------------------------------------
    # FIGURE 2: Precision-Recall Curve (AUPRC)
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=fig_size)
    baseline = df['y_true'].sum() / len(df['y_true'])
    ax.plot([0, 1], [baseline, baseline], linestyle='--', color='#7f7f7f', alpha=0.6, linewidth=1.5, label=f'Baseline ({baseline:.2f})')

    for model_name in unique_models:
        m_df = df[df['Model'] == model_name]
        precision, recall, _ = precision_recall_curve(m_df['y_true'], m_df['y_pred'])
        auprc = average_precision_score(m_df['y_true'], m_df['y_pred'])
        
        color = MODEL_COLORS.get(model_name, '#000000')
        linewidth = 2.5 if model_name == 'stacking' else 1.8
        
        ax.plot(recall, precision, color=color, lw=linewidth,
                label=f'{model_name.upper()} (AUPRC = {auprc:.3f})')
        
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel('Recall (Sensitivity)', labelpad=10)
    ax.set_ylabel('Precision (Positive Predictive Value)', labelpad=10)
    ax.set_title('Precision-Recall Curve (PRC)', weight='bold', pad=15)
    
    # Shifted slightly away from the left edge and up from the bottom line
    ax.legend(loc='lower left', bbox_to_anchor=(0.02, 0.08))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)
    
    plt.savefig('fig_precision_recall_curve.png')
    plt.savefig('fig_precision_recall_curve.pdf', format='pdf')
    plt.close(fig)
    print("Saved: fig_precision_recall_curve.png/.pdf")

    # -------------------------------------------------------------------------
    # FIGURE 3: Calibration Curve
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=fig_size)
    ax.plot([0, 1], [0, 1], linestyle='--', color='#7f7f7f', alpha=0.6, linewidth=1.5, label='Perfect Calibration')

    for model_name in unique_models:
        m_df = df[df['Model'] == model_name]
        prob_true, prob_pred = calibration_curve(m_df['y_true'], m_df['y_pred'], n_bins=10, strategy='uniform')
        brier = brier_score_loss(m_df['y_true'], m_df['y_pred'])
        
        color = MODEL_COLORS.get(model_name, '#000000')
        linewidth = 2.0 if model_name == 'stacking' else 1.4
        marker_size = 5 if model_name == 'stacking' else 4
        
        ax.plot(prob_pred, prob_true, marker='o', markersize=marker_size, color=color, lw=linewidth,
                label=f'{model_name.upper()} (Brier = {brier:.3f})')
        
    ax.set_xlim([-0.01, 1.01])
    ax.set_ylim([-0.01, 1.01])
    ax.set_xlabel('Mean Predicted Probability', labelpad=10)
    ax.set_ylabel('Fraction of Positives', labelpad=10)
    ax.set_title('Calibration Curve', weight='bold', pad=15)
    
    # Shifted slightly down from the absolute top ceiling to sit beautifully in the upper-left quadrant
    ax.legend(loc='upper left', bbox_to_anchor=(0.02, 0.96))
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='both', linestyle=':', alpha=0.4)
    
    plt.savefig('fig_calibration_curve.png')
    plt.savefig('fig_calibration_curve.pdf', format='pdf')
    plt.close(fig)
    print("Saved: fig_calibration_curve.png/.pdf")

# ==========================================
# 3. RUN EXECUTION
# ==========================================
csv_file_path = './all_test_predictions.csv' 
generate_individual_plots(csv_file_path)